<center>

# **DeepTris**

Autores:

Jordi Hamberg Gallego

Adriana González Gómez 

Héctor Sancho Rodríguez

<img src="media/logo.png" width="400"/>

*Agente de aprendizaje por refuerzo para jugar al Tetris.*

</center>

## Motivación del problema

El Tetris es el juego más vendido de la historia, y es reconocido en todo el mundo. Se juega en un tablero bidimensional de 20 x 10, y consiste en colocar piezas diferentes (llamadas *tetrominós*) mediante rotaciones y movimientos horizontales. El objetivo es completar líneas horizontales, de manera que las filas completadas se eliminen del tablero, y el jugador pueda intentar aguantar hasta que las piezas desborden el tablero por la parte de arriba.

<center>
<img src="https://teaching.csse.uwa.edu.au/units/CITS4211/Project/13/Pieces.png">
</center>

Desde hace tiempo se llevan usando técnicas de aprendizaje por refuerzo para aprender a jugar a videojuegos. Nos hemos basado especialmente en esto.

El artículo de OpenAI publicado en 2013 titulado:

[Playing Atari with Deep Reinforcement Learning](https://arxiv.org/abs/1312.5602)

que introdujo:
- DQNs
- Experience Replay
- Target Network

Aunque en siguientes artículos, el mismo autor introdujo el resto de tecnologías que hemos estudiado como:

- A3C
- CNNs para que el agente infiera cómo extraer información de los estados sin tener que hacer state-engineering.

Con lo cual, sabemos que estas técnicas son indicadas para trabajar con este tipo de juegos.

Por supuesto, resolver problemas de control con redes neuronales y un state-engineering ligero tiene transferencia a muchas áreas de la IA como la robótica y la conducción autónoma.

Además, el juego requiere planificación y estrategia, y este tipo de métodos RL son conocidos por encontrar soluciones antiintuitivas en este tipo de escenarios, así que esperamos conseguir resultados que demuestren un cierto grado de estrategia en el agente.

## **Modelado del Problema**

A continuación vamos a hablar sobre las decisiones de diseño que hemos hecho para modelar el Tetris como un Proceso de Decisión de Markov o MDP.

El objetivo del agente es aprender la política que coloque las piezas de forma eficiente, maximizando el número de líneas completadas (el objetivo principal de Tetris) e intentando no perder (cuando no caben las piezas).

Un MDP se modela mediante la tupla (S, A, P, R, γ). 

Aquí, S representa el espacio de estados, A el espacio de acciones, P la dinámica de transición, R la función de recompensa y γ el factor de descuento.

### **Espacio de Estados (S)**
El objetivo del espacio de estados es dar la información necesaria para que el agente decida la acción que va a tomar en cada instante.

Para representar el Tetris lo hemos implementado como un entorno de Gymnasium. Inicialmente, lo codificamos solamente con tres elementos, con los que conseguiremos también las *features* derivadas/adicionales del *state engineering*: el tablero, la pieza actual y la siguiente pieza.

El tablero lo modelamos como una matriz binaria de **tamaño 20x10**. Hemos elegido estas dimensiones porque es el tamaño original del Tetris. Cada una de las posiciones del tablero tendrá un número asignado:

- Si en la celda hay un **0**, significa que está vacía (no hay ni una pieza ya colocada ni la pieza que estamos colocando).
- Si hay un **1**, es porque en esa celda hay una parte de una pieza ya colocada.

Así podemos entender qué partes del tablero están libres y cuáles ocupadas.
 
Las piezas que hemos incluido son las clásicas del Tetris, que originalmente se llaman S, Z, I, O, J, L, T. También les hemos incluido las rotaciones que permite hacer el juego.

Aunque este es el contexto que le pasamos al environment, a los algoritmos realmente le vamos a pasar otras *features* que extraemos a partir de él. 

Hemos construido un vector de *features* que da una visión general del estado del juego con menos valores. Para la creación de algunas *features* nos hemos inspirado en [este artículo](https://codemyroad.wordpress.com/2013/04/14/tetris-ai-the-near-perfect-player/) que resuelve el Tetris mediante heurísticas.

Las *features* son las siguientes:
- La altura de cada una de las columnas (10). Principalmente, dan información sobre qué tan cerca del límite superior se encuentran las columnas.
- El número de huecos en cada columna (10). El principal problema que nos dan los huecos es que tenemos que completar líneas en esa columna antes de poder acceder a ellos. Además, los huecos evitan que podamos completar las líneas que están a su altura hasta que podamos acceder a ellos. Por eso queremos que se minimicen.
- La profundidad de los pozos en cada columna (10). Esto es para facilitar información sobre dónde colocar piezas como la I, la J o la L.
- La irregularidad de la superficie del tablero o *bumpiness* (1). Es simplemente la suma de las variaciones de alturas de las columnas, esperando que aprenda a minimizarlo para no dejar huecos gigantes por todas partes.
- Un *one-hot encoding* de la pieza actual (7).
- Un *one-hot encoding* de la pieza siguiente (7).

De esta manera nos quedamos con un vector de 45 valores.
Hemos tomado la decisión de hacer este *feature engineering* para reducir la complejidad del problema y también porque creemos que de esta manera se facilita más el aprendizaje que si le pasáramos las 200 celdas binarias. Trabajar con tantas celdas sería muy costoso.

En cuanto a si es continuo o no, el juego original ya es un tablero discreto y la cantidad de piezas posibles a poner también es finita. Al sacar estas *features* del tablero original, el vector también es discreto.

### **Espacio de Acciones (A)**
En el entorno al principio lo hemos codificado como que puede hacer 6 acciones:
- 0: Rota la pieza en sentido antihorario
- 1: Rota la pieza en sentido horario
- 2: Mueve la pieza hacia la izquierda
- 3: Mueve la pieza hacia la derecha
- 4: Baja la pieza una fila
- 5: Hace un *hard drop*, es decir, deja caer la pieza al fondo directamente.

De esta manera, podemos controlar completamente nuestra pieza sin limitaciones respecto al juego original, ya que puede decidir cómo colocarla donde quiera con movimientos legales.

Para el DQN, hemos hecho un ligero cambio. En lugar de hacer que la red decida para cada paso la acción a elegir, hemos hecho que elija una combinación que devuelve una tupla (columna objetivo, rotación). Hay 10 posibles columnas y 4 posibles rotaciones, por lo que el número de acciones es de 40 (cada combinación posible). Algunas combinaciones puede que no sean válidas según el estado y la pieza que tengamos, ya que puede que se produzcan colisiones. En esos casos, el movimiento que colisionaría no se ejecutaría y el agente supuestamente aprenderá estos errores.

Esto luego se lo pasamos a una función que se encarga de traducir esa tupla en la secuencia de movimientos hasta la posición deseada. 

De esta manera, simplificamos el aprendizaje, ya que la red neuronal no tiene que aprender cada paso intermedio. Solamente tiene que aprender dónde colocar cada pieza en cada paso. También hace que el aprendizaje sea mucho más rápido de esta manera.

### **Función de Recompensa (R)**

Esto determina cómo aprende el agente. Al principio pensamos en solamente dar recompensa al completar líneas, pero creímos que de esta manera no intentaría minimizar algunos aspectos como el *bumpiness* del que hemos hablado o la altura a la que llegan las líneas.

Por eso, al final hemos hecho una recompensa conjunta con los siguientes componentes:
- Recompensa positiva por seguir vivo (+1)
- Recompensa positiva por completar líneas (+10 por línea)
- Penalización por crear nuevos huecos (-4 por hueco creado en este movimiento)
- Penalización por aumentar demasiado la altura del tablero (-0.5 por la diferencia de alturas entre este movimiento y el anterior)
- Penalización por aumentar el *bumpiness* (-0.3 por diferencia de irregularidad)
- Penalización muy fuerte si muere y acaba la partida (-50)

La recompensa por seguir vivo es para que también intente sobrevivir y no solamente maximizar la puntuación. 

El número de líneas completadas, como es algo que ocurre raramente, le damos una recompensa bastante mayor. Aun así, no damos una recompensa muy exagerada para que también tenga en cuenta los posibles peligros al completar la línea.

Las alturas y el *bumpiness* no lo penalizamos mucho porque son compuestas (puede multiplicarse varias veces la penalización en cada movimiento). Los penalizamos porque debería estar lo más lejos posible del final del juego y porque es más difícil colocar piezas en superficies muy irregulares. 

La de crear nuevos huecos, como se trata de un error bastante grave, hemos decidido ponerle una penalización mediana, porque al final nos quita oportunidades de completar nuevas líneas y dificulta el juego.

Finalmente, la justificación de semejante penalización al acabarse la partida es bastante lógica, ya que es algo que sucede muy poco pero no queremos que pase bajo ningún concepto.

### **Dinámica de Transición (P)**
Aquí indicamos cómo cambia el estado del juego después de cada paso elegido por nuestro agente.

La transición es determinista en lo que respecta a las reglas del juego. Si el agente decide que irá a un sitio con una rotación, la pieza lo hará siempre que no tenga ninguna pared en medio que le impida hacerlo (que en ese caso no se hará ese movimiento). Es decir, se sabe más o menos cómo será la disposición del tablero después del movimiento y la pieza que tendremos en el siguiente paso.

Sin embargo, tenemos una parte estocástica, ya que la "siguiente pieza" del siguiente paso sí que se genera aleatoriamente de entre las 7 posibles. Entonces, aunque el paso siguiente sí que sea determinista, no puede controlar cuál será su pieza en los 2 pasos siguientes en adelante.

El problema es episódico, en el que cada episodio empieza con el tablero vacío y tiene un final de partida, (cuando el tablero está tan lleno que no podemos generar una nueva pieza).

### **Factor de Descuento (γ)**
Aquí indicamos cuánto peso tienen las recompensas futuras en nuestra función de recompensa.

En nuestro caso hemos elegido un factor de 0.99. Hemos elegido este valor porque al final tienes que hacer una serie de pasos para llegar a completar una sola línea. Esto hace que aunque dos colocaciones de la misma pieza en diferentes posiciones no completen ninguna línea, una tenga mayor recompensa si hace que el tablero esté más cerca de completar líneas en futuros movimientos.

Si pusiéramos al agente con un factor de descuento cercano a 0, no valoraría bien la colocación de las piezas para completar líneas futuras e intentaría minimizar la penalización, dejando muy probablemente huecos de por medio en cada paso.

Un factor de descuento de 1 tampoco sería perfecto, ya que no tenemos conocimiento completo sobre las piezas futuras que tendremos.

## Breve estado del arte

Durante el desarrollo del proyecto hemos encontrado varios artículos relacionados con el tema que implementan soluciones de diverso tipo para el problema de jugar al Tetris. Es un problema muy estudiado especialmente porque es simple pero la cantidad de estados es muy grande por la naturaleza combinatoria del tablero y las piezas (~$2^{10 \times 20}$ posibles tableros); además, se ha demostrado que es un problema NP-completo. Algunos de los enfoques que hemos encontrado de personas que han resuelto el problema de jugar al Tetris son:

### Enfoques heurísticos

Nos ha sorprendido mucho que hemos encontrado enfoques muy modernos y muy exitosos completamente basados en reglas y algoritmos de búsqueda. El más conocido es el controlador de [Pierre Dellacherie](https://github.com/yanyongyu/python-tetris) que se basa en una función de evaluación del tablero con 6 características:

$$Score=−Landing Height+Eroded Piece Cells−Row Transitions−Column Transitions−4⋅Holes−Cumulative Wells$$

Con esta función, la cual ajustó a base de prueba y error, pudo usar una búsqueda sencilla y fue capaz de conseguir una media de 660_000. Más adelante se probaron nuevas heurísticas y sobre todo métodos de optimización de los pesos de la función de evaluación, y se consiguieron resultados de hasta 35_000_000 de puntos, logrando superar métodos de aprendizaje por refuerzo.

### Aprendizaje por refuerzo clásico

No hemos podido encontrar muchos artículos que usen métodos de aprendizaje por refuerzo clásico; parece que el interés por este problema creció a partir de la introducción de los DQNs, y no se han publicado muchos artículos usando métodos clásicos.

#### DQNs y A3C

Estos son los artículos que más hemos consultado para el desarrollo de nuestro proyecto, especialmente:

- [Playing Tetris with Deep Reinforcement Learning](https://cs231n.stanford.edu/reports/2016/pdfs/121_Report.pdf)

En este se explica el uso de una CNN que consigue extraer características del tablero.

- [Learn to Play Tetris with Deep Reinforcement Learning](https://openreview.net/pdf?id=8TLyqLGQ7Tg)

En este se comparan muchos métodos y se concluye que DQN es el mejor método para este problema.

<center><img src="media/resultados_paper.png"  width="700"/></center>

#### Mejores implementaciones basadas en RL

Algunas de las implementaciones más exitosas que hemos encontrado son:

#### Mejor récord actual

En marzo de 2026 se publicó [Bitboard version of Tetris AI](https://arxiv.org/abs/2603.26765) que hace varios cambios relevantes y consigue entrenamientos rapidísimos:

- Construyeron un motor que aprovechaba la representación del tablero para explotar las operaciones bit a bit, lo que le permitió conseguir velocidades de entrenamiento unas 53 veces más rápidas que una implementación tradicional.
- La red evaluaba los *afterstates*, es decir, el estado resultante de aplicar una acción.
- Proponen un algoritmo PPO con buffer optimizado que balancea eficiencia de muestreo y actualización, alcanzando una puntuación media de 3.829 líneas en tableros 10×10 en tan solo 3 minutos de entrenamiento. 

#### Implementación en hardware real

En 2022 se publicó [Deep reinforcement learning in playing Tetris with robotic arm experiment](https://journals.sagepub.com/doi/10.1177/01423312221114694) donde se implementa un agente que usa aprendizaje por refuerzo pero interactuando directamente con el hardware real a través de una cámara y un brazo robótico.

#### Algoritmos generalistas

<center>
<img src="https://lh3.googleusercontent.com/Gy24iLqDpKl4TyH4BcMCFNOkiDlMRg6PbclXrOqhp6stgd8dQZHTabSqonlYa5UOZcv0EcGPhVS0DQK5ZEkFNHkJUom24m1__jIlRvXqkmaTCUOb=w1440-rw-lo"  width="700"/>
</center>

Por último, el aprendizaje profundo logró en 2020 conseguir un agente capaz de dominar el ajedrez, el go, el shogi y los 57 juegos de Atari, y entre ellos el Tetris, sin ninguna información específica del juego, de manera que el modelo infiera también las normas del juego: [MuZero: Mastering Go, chess, shogi and Atari without rules](https://deepmind.google/blog/muzero-mastering-go-chess-shogi-and-atari-without-rules/)

## Solución propuesta

La primera solución que implementamos es Double DQN con Experience Replay. A continuación detallamos los aspectos más relevantes de la implementación.

In [14]:
from enviroment import TetrisEnv
import numpy as np
import torch

### Traducción a nuestras acciones

Como hemos comentado en el modelado del problema, hemos mapeado cada acción (una por pieza) a una secuencia de inputs que le pasaremos al entorno para colocar la pieza en la posición.

In [15]:
# Espacios de acción: 0=rotar_izq, 1=rotar_der, 2=izq, 3=der, 4=bajar, 5=hard_drop

# El resultado de la red neuronal sera una tupla (0-9 columnas, 0-3 rotaciones)
def action_move_sequence(action : tuple((int, int))) -> tuple:
    sequence = []

    column = action[0] - 5
    rotation = action[1] - 1

    if rotation < 0:
        for i in range(-rotation):
            sequence.append(0)
    else:
        for i in range(rotation):
            sequence.append(1)

    if column < 0:
        for i in range(-column):
            sequence.append(2)
    else:
        for i in range(column):
            sequence.append(3)

    sequence.append(5)

    return sequence

### Arquitectura de la red neuronal (Agente)

Implementamos una red neuronal que tomará como entrada un estado codificado y devolverá los valores Q para cada acción posible. Una vez identificada la acción con el mayor valor Q, traduciremos esa acción a la secuencia de inputs que le pasaremos al entorno para colocar la pieza en la posición. 

La arquitectura tiene 4 capas ocultas con [256, 256, 256, 128] neuronas respectivamente, con activación ReLU. La capa de salida tiene 40 neuronas, una por cada acción posible (columna objetivo, rotación).

Hemos añadido varias regularizaciones como *dropout* y *batch normalization* para evitar el *overfitting* y mejorar la generalización del modelo. Uno de los desafíos más importantes de este problema es evitar el *overfitting*, ya que muchos estados son similares, y esto puede evitar que el modelo generalice bien a estados nuevos. El uso de Double DQN y *experience replay* también ayuda mucho a mejorar la estabilidad en este aspecto.

**Política $\epsilon$-greedy**: La política de acción que hemos implementado es $\epsilon$-greedy, lo que significa que con una probabilidad $\epsilon$ el agente elegirá una acción aleatoria (exploración), y con una probabilidad $1-\epsilon$ elegirá la acción con el mayor valor Q (explotación). Esto permite al agente explorar el espacio de acciones y evitar quedarse atrapado en soluciones subóptimas. El valor de $\epsilon$ no es fijo, sino que se puede ir reduciendo en el bucle de entrenamiento, cosa que hacemos para que el agente explore más al principio y luego se vaya centrando en la explotación a medida que va aprendiendo.

In [16]:
from torch import nn
import random

N_ACCIONES = 40

class DQN_Agent(nn.Module):
    def __init__(self, layers_sizes = [45, 256, 256, 256, 128], dropout_value = 0.0):
        super().__init__()
        self.dropout_value = dropout_value
        layers = []

        for i in range(len(layers_sizes) - 1):
            current_size = layers_sizes[i]
            next_size = layers_sizes[i + 1]

            layers.extend([
                torch.nn.Linear(current_size, next_size),
                torch.nn.LayerNorm(next_size),
                torch.nn.ReLU(),
            ])
            if self.dropout_value > 0:
                layers.append(torch.nn.Dropout(self.dropout_value))
        
        self.encoder = torch.nn.Sequential(*layers)
        self.head_q = torch.nn.Linear(layers_sizes[-1], N_ACCIONES)

    def forward(self, state: list) -> torch.Tensor:
        state_tensor = torch.tensor(state, dtype=torch.float32).to(device)
        encoded_state = self.encoder(state_tensor)
        q_values = self.head_q(encoded_state)
        return q_values
    
    def act(self, state, epsilon=0.0):
        if random.random() < epsilon:
            action_index = random.randint(0, N_ACCIONES - 1)
        else:
            with torch.no_grad():
                q_values = self.forward(state)
                action_index = torch.argmax(q_values).item()

        column = action_index // 4
        rotation = action_index % 4

        action = action_move_sequence((column, rotation))

        return action_index, action

### Creación de funciones auxiliares 
También hemos hecho funciones auxiliares para la extracción de features.

La primera que hemos hecho es la de obtener las alturas de cada una de las columnas. Simplemente va por cada una de ellas y se queda con la posición más alta que encuentra.

In [17]:
def _get_heights(board):
    n_rows, n_cols = board.shape
    heights = np.zeros(n_cols, dtype=np.int32)
    for col in range(n_cols):
        occupied = np.where(board[:, col] == 1)[0]
        if len(occupied) > 0:
            heights[col] = n_rows - occupied[0]
    return heights


La segunda es la de contar la cantidad de huecos que hay en cada columna. Mira cuántas celdas vacías hay debajo de la altura máxima de cada una de las columnas.

In [18]:

def _get_holes(board, heights):
    n_rows, n_cols = board.shape
    holes = np.zeros(n_cols, dtype=np.int32)
    for col in range(n_cols):
        if heights[col] > 0:
            top_row = n_rows - heights[col]
            column_data = board[top_row:, col]
            holes[col] = np.sum(column_data == 0)
    return holes


La tercera es la bumpiness o irregularidad del terreno. Es un solo valor y se trata de la suma de las diferencias de las alturas de columnas contiguas.

In [19]:
def _get_bumpiness(heights):
    n_cols = len(heights)
    bumpiness = 0
    if n_cols > 1:
        bumpiness = np.sum(np.abs(np.diff(heights)))
    return bumpiness


La cuarta es la de mirar los pozos que hay en cada una de las columnas. Compara la altura de cada una de las columnas vecinas, mira cuál es la mínima de entre ellas y hace la resta entre ese valor y la altura de la columna en la que estamos. De esta manera, vemos cuántos espacios libres podríamos rellenar con piezas.

In [20]:
def _get_wells(heights):
    n_cols = len(heights)
    wells = np.zeros(n_cols, dtype=np.int32)
    for col in range(n_cols):
        left_height = heights[col - 1] if col > 0 else heights[col]
        right_height = heights[col + 1] if col < n_cols - 1 else heights[col]

        min_neighbor = min(left_height, right_height)
        if heights[col] < min_neighbor:
            wells[col] = min_neighbor - heights[col]
    return wells


La quinta es el one hot de las piezas actual y siguiente. Simplemente crea dos vectores vacíos de 7 (el número de piezas diferentes que tenemos en el juego) y le pone un 1 en la posición que tiene asignada ese tipo de pieza.

In [21]:

def _get_onehot_pieces(state):
    current_piece_onehot = np.zeros(7, dtype=np.float32)
    current_piece_onehot[state['current_piece']] = 1

    next_piece_onehot = np.zeros(7, dtype=np.float32)
    next_piece_onehot[state['next_piece']] = 1
    return current_piece_onehot, next_piece_onehot


Por último, hemos creado una función que nos crea el vector completo de todas las features. Además normaliza las alturas, los huecos, los pozos y el bumpiness para que esté entre 0 y 1 y así aprenda mejor la red neuronal.

In [22]:

def extract_tetris_features(state):
    """
    Extrae características del estado de Tetris para aprendizaje por refuerzo.
    """
    board = state['board']

    heights = _get_heights(board)
    holes = _get_holes(board, heights)
    bumpiness = _get_bumpiness(heights)
    wells = _get_wells(heights)
    current_piece_onehot, next_piece_onehot = _get_onehot_pieces(state)

    max_h = float(board.shape[0])
    feature_vector = np.concatenate([
        heights / max_h,
        holes / max_h,
        wells / max_h,
        current_piece_onehot,
        next_piece_onehot,
        [bumpiness / max_h]
    ])

    return feature_vector

### Optimización con acelerador GPU

Como contamos con una GPU, hemos hecho que el entrenamiento se realice en pequeños *batches* y que se realice en la GPU, lo que nos ha permitido acelerar el entrenamiento de manera significativa. Hemos comprobado que solo usar la GPU da malos resultados porque el *overhead* de pasar los datos a la GPU cada vez que quieres hacer una ronda de entrenamiento es muy alto; por eso, lo que hacemos es acumular experiencias en el buffer de *experience replay*, y luego cada cierto número de pasos hacemos un *batch* de entrenamiento con la GPU. De esta manera conseguimos acelerar el entrenamiento de manera significativa.

In [23]:
from torch.cuda import get_device_name, is_available

print("CUDA disponible:", is_available())
if is_available():
    print("Dispositivo CUDA:", get_device_name(0))

device = torch.device("cuda" if is_available() else "cpu")

CUDA disponible: True
Dispositivo CUDA: NVIDIA GeForce RTX 2060


### Parámetros de entrenamiento

- Optimización: Adam con una tasa de aprendizaje de 1e-4 (muy estándar).
- Loss: SmoothL1Loss, que es una función de pérdida robusta a *outliers*, lo que puede ser útil en este tipo de problemas donde las recompensas pueden variar mucho.
- Epsilon: Empezamos con un valor de 1 para fomentar la exploración, y lo vamos reduciendo gradualmente a medida que el agente aprende, hasta un mínimo de 0.05 para asegurar que siempre haya algo de exploración.

In [24]:
from torch import optim

agent = DQN_Agent().to(device)
optimizer = optim.Adam(agent.parameters(), lr=1e-4)
loss_fn = nn.SmoothL1Loss()

### Iteración de entrenamiento en batch

A partir de un *batch* de 128 experiencias del buffer (state, action, reward, next_state, done), realizamos los siguientes pasos:

1. Recalculamos los valores Q calculados por el agente principal (usando la red principal).
2. Calculamos los valores Q del target (usando la red target y la recompensa empírica):
$$Q_{target} = r + \gamma \max_{a'} Q_{target}(s', a')$$

3. Calcular la pérdida. La lógica de esta parte es que Q_principal debería seguir las ecuaciones de Bellman y, por tanto, debería ser igual a Q_target; por eso calculamos la pérdida entre ambos:

$$Loss = SmoothL1Loss(Q_{principal}, Q_{target})$$

4. Actualizar pesos (usando *backpropagation* y el optimizador Adam).

In [25]:
import torch.optim as optim
import torch.nn.functional as F
import torch


def train_dqn(agent, target_agent, optimizer, loss_fn, batch, gamma):

    states = torch.tensor(np.array([exp[0] for exp in batch]), dtype=torch.float32).to(device)
    actions = torch.tensor([exp[1] for exp in batch], dtype=torch.int64).unsqueeze(1).to(device)
    rewards = torch.tensor([exp[2] for exp in batch], dtype=torch.float32).to(device)
    next_states = torch.tensor(np.array([exp[3] for exp in batch]), dtype=torch.float32).to(device)
    dones = torch.tensor([exp[4] for exp in batch], dtype=torch.float32).to(device) 
    
    q_values = agent(states)
    current_q = q_values.gather(1, actions).squeeze(1)
    
    with torch.no_grad():
        best_actions = agent(next_states).argmax(dim=1, keepdim=True)
        
        next_q_values = target_agent(next_states)
        max_next_q = next_q_values.gather(1, best_actions).squeeze(1)
        
        target_q = rewards + gamma * max_next_q * (1 - dones)
    
    loss = loss_fn(current_q, target_q)
    
    optimizer.zero_grad()
    loss.backward()
    
    torch.nn.utils.clip_grad_norm_(agent.parameters(), max_norm=1.0)
    
    optimizer.step()
    
    return loss.item()

### Training Loop

Esta es la parte final del entrenamiento, en la que se pone el modelo a funcionar y empieza el aprendizaje.

Primero de todo hemos definido unos parámetros:
- Experience Replay: 50000. Hemos puesto este número para que nuestro aprendizaje tenga un buffer suficientemente grande para poder guardar varias partidas (de hecho muchas partidas al principio del aprendizaje) y tenga una muestra lo suficientemente grande como para no overfittear.
- Tamaño del batch de entrenamiento y frecuencia de entrenamiento con el batch: 128 y 4. Hemos puesto un batch de 128 para que el overhead de cargar los datos a la gráfica valga la pena. Hacer un batch más pequeño desperdiciaría el tiempo que ya le dedicamos a pasar esos datos a la VRAM.
- Gamma: 0.99. Suficiente para que el agente valore las recompensas futuras pero sin perder de vista las recompensas inmediatas (ya explicado en el modelado del problema).
- Actualización de los pesos de la red objetivo: 1000. Para que no se produzca el fenómeno de perseguirse la cola y para que a la red no le cueste tanto converger.


In [26]:
N_EXPERIENCE_REPLAY = 50000
batch_train_size = 128
train_frequency = 4
gamma = 0.99
TARGET_UPDATE_FREQ = 1000

En cuanto a la estrategia epsilon greedy, el proceso empieza con una exploración absoluta de 1.0, porque al principio el agente no tiene ni idea de cómo jugar. 

Decae lentamente hasta 0.05, valor que a partir de ahí no baja para mantener siempre algo de exploración aunque casi siempre recurra a la explotación.

El decaimiento se hace cada 500.000 pasos. Al principio pusimos 100.000, pero decaía muy rápido y no le daba tiempo a explorar.

In [27]:
# PARÁMETROS EPSILON-GREEDY
epsilon_start = 1.0
epsilon_end = 0.05
epsilon_decay_steps = 500000

Para el experience replay hemos utilizado la estructura de datos deque porque así sigue la estrategia FIFO, olvidándose de las recompensas de las decisiones que tomó en momentos más alejados a la actualidad. En esas decisiones, la arquitectura de la red neuronal probablemente era muy diferente a la del instante actual, así que quizás el agente ya ni siquiera considera el tomar una decisión como la que tomó en ese instante.

In [28]:
from collections import deque
experience_replay = deque(maxlen=N_EXPERIENCE_REPLAY)

También hemos creado unas variables para el logging y para hacer las gráficas más adelante. 

In [29]:
log_frequency = 5000
episode_rewards_log = []
loss_log = []

Ahora cargamos la red objetivo y la ponemos en modo eval para que no se actualicen los pesos a no ser que lo indiquemos nosotros.

In [30]:
target_agent = DQN_Agent().to(device)
target_agent.load_state_dict(agent.state_dict())
target_agent.eval()

DQN_Agent(
  (encoder): Sequential(
    (0): Linear(in_features=45, out_features=256, bias=True)
    (1): LayerNorm((256,), eps=1e-05, elementwise_affine=True, bias=True)
    (2): ReLU()
    (3): Linear(in_features=256, out_features=256, bias=True)
    (4): LayerNorm((256,), eps=1e-05, elementwise_affine=True, bias=True)
    (5): ReLU()
    (6): Linear(in_features=256, out_features=256, bias=True)
    (7): LayerNorm((256,), eps=1e-05, elementwise_affine=True, bias=True)
    (8): ReLU()
    (9): Linear(in_features=256, out_features=128, bias=True)
    (10): LayerNorm((128,), eps=1e-05, elementwise_affine=True, bias=True)
    (11): ReLU()
  )
  (head_q): Linear(in_features=128, out_features=40, bias=True)
)

Aquí ya empieza el bucle de entrenamiento:

1. Por cada paso elige el epsilon y la acción. 
2. Una vez elegida, mira cuántas líneas se han completado o si ha terminado la partida.
3. Si no ha terminado la partida, compara el feature state antes y después de la acción elegida y calcula la función de recompensa que creamos.
4. Se suma la función de recompensa del paso a la del episodio total, también se añade la experiencia al Experience Replay y entrena con el batch de experiencias si puede y si toca.
5. Cuando toca, también actualiza los parámetros de la red objetivo a los de la red principal (la de comportamiento).
6. Se repite hasta que termina el episodio.


In [31]:
import random
from time import sleep
from collections import deque
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import clear_output
import copy

N_EXPERIENCE_REPLAY = 50000   
batch_train_size = 128        
train_frequency = 4           
gamma = 0.99                  
TARGET_UPDATE_FREQ = 1000     

epsilon_start = 1.0
epsilon_end = 0.05
epsilon_decay_steps = 500000  

experience_replay = deque(maxlen=N_EXPERIENCE_REPLAY)
steps_counter = 0

log_frequency = 5000  # Frecuencia de actualización de gráficas
episode_rewards_log = []
loss_log = []

target_agent = DQN_Agent().to(device)
target_agent.eval()

for episode in range(10000000):

    env = TetrisEnv(n=20, m=10)

    state, info = env.reset()
    current_feature_vector = extract_tetris_features(state)

    terminated = False
    truncated = False
    current_episode_reward = 0
    current_loss = 0

    while not (terminated or truncated):
        
        epsilon = max(epsilon_end, epsilon_start - (epsilon_start - epsilon_end) * (steps_counter / epsilon_decay_steps))
        
        action_index, action_sequence = agent.act(current_feature_vector, epsilon=epsilon)
        steps_counter += 1

        lines_cleared = 0
        for a in action_sequence:
            state, _, terminated, truncated, info = env.step(a)
            lines_cleared += info.get('step_lines_cleared', 0)
            if terminated or truncated:
                break
                
        next_feature_vector = extract_tetris_features(state)

        new_height = np.sum(next_feature_vector[0:10]) * 20.0
        new_holes = np.sum(next_feature_vector[10:20]) * 20.0
        new_bumpiness = next_feature_vector[44] * 20.0
        
        old_height = np.sum(current_feature_vector[0:10]) * 20.0
        old_holes = np.sum(current_feature_vector[10:20]) * 20.0
        old_bumpiness = current_feature_vector[44] * 20.0
        
        delta_holes = new_holes - old_holes
        delta_height = new_height - old_height
        delta_bumpiness = new_bumpiness - old_bumpiness
        
        alive_reward = 1 
        game_over_penalty = -50 if terminated else 0
        
        custom_reward = (
            alive_reward + 
            10 * lines_cleared +
            -4 * delta_holes +         
            -0.5 * delta_height +      
            -0.3 * delta_bumpiness +   
            game_over_penalty
        )
        
        current_episode_reward += custom_reward

        experience_replay.append((current_feature_vector, action_index, custom_reward, next_feature_vector, terminated))

        if len(experience_replay) >= batch_train_size and steps_counter % train_frequency == 0:
            batch_indices = random.sample(range(len(experience_replay)), batch_train_size)
            batch = [experience_replay[i] for i in batch_indices]
            
            loss = train_dqn(agent, target_agent, optimizer, loss_fn, batch, gamma)  
            current_loss = loss      

        if steps_counter % TARGET_UPDATE_FREQ == 0:
            target_agent.load_state_dict(agent.state_dict())

        current_feature_vector = next_feature_vector

    episode_rewards_log.append(current_episode_reward)
    loss_log.append(current_loss)
    
    if (episode + 1) % log_frequency == 0:
        avg_reward = np.mean(episode_rewards_log[-log_frequency:])
        
        clear_output(wait=True)
        
        max_points = 20000
        if len(episode_rewards_log) > max_points:
            step = len(episode_rewards_log) // max_points
            rewards_plot = episode_rewards_log[::step]
            loss_plot = loss_log[::step]
            x_axis = np.arange(len(rewards_plot)) * step
        else:
            rewards_plot = episode_rewards_log
            loss_plot = loss_log
            x_axis = np.arange(len(episode_rewards_log))
        
        fig, axes = plt.subplots(1, 2, figsize=(15, 5))
        
        axes[0].plot(x_axis, rewards_plot, label='Recompensa', alpha=0.3)
        if len(rewards_plot) >= 50:
            running_avg = [np.mean(rewards_plot[max(0, i-50):i+1]) for i in range(len(rewards_plot))]
            axes[0].plot(x_axis, running_avg, color='red', label='Media Móvil')
            
        axes[0].set_title('Recompensa por Episodio')
        axes[0].set_xlabel('Episodio')
        axes[0].set_ylabel('Recompensa')
        axes[0].legend()
        
        axes[1].plot(x_axis, loss_plot, label='Pérdida (Loss)', color='orange')
        axes[1].set_title('Pérdida a lo largo de los episodios')
        axes[1].set_xlabel('Episodio')
        axes[1].set_ylabel('Pérdida')
        axes[1].legend()
        
        plt.tight_layout()
        plt.show()

        print(f"Episode {episode + 1} | Steps Totales: {steps_counter}")
        print(f"Epsilon actual: {epsilon:.3f}")
        print(f"Average Reward (last {log_frequency}): {avg_reward:.2f}")
        print(f"Last Game Score: {info.get('score', 0)} | Lines: {info.get('lines_cleared', 0)}")
        
env.close()

C:\Users\iocol\AppData\Local\Temp\ipykernel_14212\1560108995.py:28: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  state_tensor = torch.tensor(state, dtype=torch.float32).to(device)


KeyboardInterrupt: 

**Guardamos los modelos**

Puedes descomentar esta celda para guardar un modelo entrenado. Nosotros hemos entrenado un modelo 13 horas y no queremos sobrescribirlo, pero si quieres entrenar un nuevo modelo puedes descomentar esta celda para guardarlo.

In [ ]:
# torch.save(agent.state_dict(), "dqn_tetris_agent.pth")
# torch.save(target_agent.state_dict(), "dqn_tetris_target_agent.pth")

Cargamos un modelo entrenado.

In [ ]:
agent.load_state_dict(torch.load("dqn_tetris_agent.pth", map_location=device))
target_agent.load_state_dict(torch.load("dqn_tetris_target_agent.pth", map_location=device))

<All keys matched successfully>

### Visualización de resultados.

In [ ]:
from time import sleep

env = TetrisEnv(n=20, m=10, render_mode='human')
state, info = env.reset()

terminated = False

for _ in range(10):  

    while not terminated:
        
        action_index, action_sequence = agent.act(extract_tetris_features(state), epsilon=0.0) 
        for a in action_sequence:
            state, _, terminated, truncated, info = env.step(a)
            sleep(0.1)  # Pequeña pausa para visualizar mejor
            env.render()
            if terminated or truncated:
                break

    

env.close()

Aquí va un ejemplo de una partida.

<video controls autoplay loop muted style="display: block; margin-left: auto; margin-right: auto;">
  <source src="media/ejemplo.mp4" type="video/mp4">
</video>